In [ ]:
import time
import numpy as np
import pandas as pd
from glob import glob
import anndata
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import cm as cm
import seaborn as sns
from scipy.sparse import csr_matrix
from ALLCools.plot import *
from ALLCools.clustering import *
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

In [ ]:
indir = '/scratch/mmoore/Epitherapy3D/analysis/HiC/paper_final/5_snm3c/schicluster/dataset/'

In [ ]:
chrom_sizes = pd.read_csv('/scratch/devtools/mmoore/genomes/snm3c/hg38/hg38.main.chrom.sort.sizes_nochrM', index_col=0, header=None, sep='\t')[1]
chrom_sizes

In [ ]:
celllist = pd.read_csv(f'{indir}../impute/25K/cell_table.tsv', sep='\t', index_col=0, header=None)
celllist

In [ ]:
meta = pd.read_csv(f'CellMetadata.PassQC.Quintile.csv.gz', sep=',', header=0, index_col=0)
meta

In [ ]:
start_time = time.time()
matrix = []
dim = 50
for chrom in chrom_sizes.index:
    tmp = np.load(f'{indir}embedding/raw/{chrom}.npz')['arr_0']
    dim = min(dim, tmp.shape[0] - 1, tmp.shape[1] - 1)
    model = TruncatedSVD(n_components=dim, algorithm='arpack')
    tmp_reduce = model.fit_transform(tmp)
    matrix.append(tmp_reduce / model.singular_values_)
    print(chrom, time.time() - start_time)

In [ ]:
model = TruncatedSVD(n_components=dim, algorithm='arpack')
matrix_reduce = model.fit_transform(np.concatenate(matrix, axis=1))
matrix_reduce = matrix_reduce / model.singular_values_

In [ ]:
adata = anndata.AnnData(X=np.ones((celllist.shape[0],1)), 
                        obs=meta.loc[celllist.index])
adata

In [ ]:
adata.obs.index.name = 'cell_name'
adata.obs['contacts'] = adata.obs['TotalContacts']

In [ ]:
adata.obs = adata.obs[['Batch', 'mCHFrac', 'mCGFrac', 'contacts','Cell_line','Treatment', 'CisLongContact','TotalContacts','R1UniqueMappedReads','quintile','qcluster']]

In [ ]:
adata.obsm['dipc_pca_all'] = matrix_reduce.copy()

In [ ]:
significant_pc_test(adata, p_cutoff=0.1, update=False, obsm='dipc_pca_all')

In [ ]:
adata.obsm['X_pca'] = normalize(adata.obsm['dipc_pca_all'][:, :20], axis=1)

In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata

In [ ]:
knn = 10
sc.pp.neighbors(adata, n_neighbors=knn, use_rep='X_pca')
sc.tl.umap(adata, maxiter=200, random_state=0)
adata = dump_embedding(adata, 'umap')

In [ ]:
sc.tl.leiden(adata, resolution=1.0, random_state=0)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='Cell_line',
                        #text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='Treatment',
                        #text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = continuous_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='mCGFrac',
                        #text_anno='leiden',
                        #show_legend=True
                      )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8), dpi=300, constrained_layout=True)
_ = categorical_scatter(ax=axes[0],
                        data=adata.obs,
                        hue='Cell_line',
                        coord_base='umap',
                        # text_anno='cell-type cluster',
                        palette='tab10',
                        #labelsize=10,
                        show_legend=True
                       )
_ = categorical_scatter(ax=axes[1],
                        data=adata.obs,
                        hue='Treatment',
                        coord_base='umap',
                        # text_anno='cell-type cluster',
                        palette='tab10',
                        #labelsize=10,
                        show_legend=True
                       )
_ = continuous_scatter(ax=axes[2],
                        data=adata.obs,
                        hue='mCGFrac',
                        coord_base='umap',
                        # text_anno='region',
                        # palette='tab10',
                        #labelsize=10
                      )
#_ = categorical_scatter(ax=axes[1,1],
#                        data=adata.obs,
#                        hue='quintile',
#                        coord_base='umap',
#                        # text_anno='cell-type cluster',
#                        palette='tab20',
#                        labelsize=10,
#                        show_legend=True
#                       )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=300, constrained_layout=True)
_ = categorical_scatter(data=adata.obs,
                        ax=ax,
                        coord_base='umap',
                        hue='leiden',
                        text_anno='leiden',
                        palette='tab20',
                        labelsize=10,
                        # show_legend=True
                       )

In [ ]:
adata.write_h5ad('GSC_25K_snm3c.h5ad')

In [ ]:
adata = sc.read_h5ad("GSC_25K_snm3c.h5ad")

In [ ]:
adata.obsm['X_umap']

In [ ]:
umap = adata.obsm["X_umap"]

print(umap.shape)

In [ ]:
from scipy.spatial.distance import pdist, squareform

dist_matrix = squareform(pdist(umap, metric="euclidean"))

print(dist_matrix.shape)

In [ ]:
import seaborn as sns
g = sns.clustermap(
    dist_matrix,
    cmap="viridis",
    row_cluster=True,
    col_cluster=True,
    #row_colors=row_colors,
    #col_colors=row_colors,
    figsize=(10,10)
)

In [ ]:
dist_df = pd.DataFrame(dist_matrix, index=adata.obs_names.values, columns=adata.obs_names.values)
dist_df.to_csv("cell_cell_euclidean_25kb.tsv", sep="\t")

In [ ]:
adata.obs_names.values